# Variant 2 — ORD only, 57,000 reactions (Model 1, Kaggle GPU)

Fine-tunes `sagawa/ReactionT5v2-retrosynthesis` on the 57,000-reaction ORD
training split with the configuration reported for variant 2: canonical SMILES
(`--no-augment`), `--learning-rate 5e-5`, three epochs, `torchrun --nproc_per_node=2`.
Identical launch recipe to notebooks 03 and 04, so variants 2, 4, 5 and 6 differ only
in training data.

57,000 examples at an effective batch of 32 is ~5,344 steps; at the 0.97 steps/s
measured on the 150k DDP run that is ~92 min, plus ~15 min for the two top-k
evaluations that follow.

Input: dataset `kuzmenkooleh/retro-planner-reactants-57k-uspto`.


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))


In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner


In [ ]:
%pip install -q -e ".[local-models,indexing]"


In [ ]:
import glob, os

# Kaggle has mounted datasets under two different layouts historically, so search
# rather than hard-code the path.
train_file = next(glob.iglob("/kaggle/input/**/reactants_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/reactants_val.jsonl", recursive=True))
uspto_file = next(glob.iglob("/kaggle/input/**/uspto_reactants_train.jsonl", recursive=True))
for path in (train_file, val_file, uspto_file):
    print(path, sum(1 for _ in open(path)), "rows")


In [ ]:
import os

output_dir = "/kaggle/working/model1_variant2_57k"
time_budget_minutes = 200  # ~92 min training + headroom
os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

!torchrun --nproc_per_node=2 scripts/train_reactant_model_ord.py \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model1_work \
    --no-augment \
    --learning-rate 5e-5 \
    --num-train-epochs 3 \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done; tail of log:")
!tail -5 "{log_path}"


In [ ]:
import os
model_dir = f"{output_dir}/final"
assert os.path.isdir(model_dir), os.listdir(output_dir)

for tag, targets in [("ord", "data/v2_ord_eval_targets.json"),
                     ("uspto", "data/v2_uspto_eval_targets.json")]:
    !python scripts/models/run_reactiont5_topk.py \
        --input "{targets}" --t5-model "{model_dir}" \
        --num-beams 10 --device cuda \
        --output "/kaggle/working/{tag}_topk.json"
    print(tag, "done")


In [ ]:
import json
for tag in ("ord", "uspto"):
    data = json.load(open(f"/kaggle/working/{tag}_topk.json"))
    print("===", tag, "===")
    print(json.dumps(data.get("summary", data), indent=2)[:800])
